# SDXL Poster LoRA — Local Fine-Tuning

Fine-tunes the **background visual** generator behind NexMarket's poster pipeline on the local RTX 3050 (6 GB).

## What this can and cannot change

The poster pipeline is two stages:
1. **SDXL** generates a background *visual* (no text).
2. **`PosterComposer`** overlays the headline, CTA, logo and brand colours with PIL.

A LoRA trained here affects **stage 1 only**. If posters look wrong because of typography, text contrast, or layout, that is stage 2 and no amount of diffusion fine-tuning will fix it.

## Hard constraints on this machine

| Constraint | Value | Consequence |
|---|---|---|
| VRAM | 6.0 GB | LoRA only — a full fine-tune needs ~40 GB |
| `bitsandbytes` | broken | no 8-bit Adam; use plain AdamW |
| torch / diffusers | 2.1.2+cu121 / 0.30.3 | pinned — this project has broken twice on version drift |

The 6 GB fit comes from **pre-computing VAE latents and text embeddings once**, then freeing the VAE and both text encoders so only the UNet occupies VRAM during training.

## Cell 1 — Environment audit

Verifies the GPU and pins down what is actually installed. Run this first; every later cell assumes it passed.

In [14]:
import sys, subprocess, importlib
import torch

print("python :", sys.version.split()[0])
print("torch  :", torch.__version__, "| cuda:", torch.cuda.is_available())

if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    VRAM_GB = p.total_memory / 1024**3
    print(f"gpu    : {p.name}  {VRAM_GB:.1f} GB")
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f"free   : {free/1024**3:.2f} GB of {total/1024**3:.2f} GB")
else:
    VRAM_GB = 0
    print("!! no CUDA device - training is not viable on CPU")

for m in ["diffusers", "transformers", "accelerate", "peft", "safetensors"]:
    try:
        print(f"{m:13s}:", importlib.import_module(m).__version__)
    except ImportError:
        print(f"{m:13s}: NOT INSTALLED")


python : 3.11.9
torch  : 2.1.2+cu121 | cuda: True
gpu    : NVIDIA GeForce RTX 3050 6GB Laptop GPU  6.0 GB
free   : 5.04 GB of 6.00 GB
diffusers    : 0.30.3
transformers : 4.49.0
accelerate   : 1.14.0
peft         : 0.13.2
safetensors  : 0.8.0


## Cell 2 — Install `peft`

`peft` provides the LoRA layers. Pinned to a release known to work with diffusers 0.30.3 / torch 2.1.2.

**This project has twice been broken by dependency drift.** Install exactly this pin, and do not let pip resolve `torch`, `numpy`, or `diffusers` to something newer.

In [15]:
import sys
!{sys.executable} -m pip install --no-deps "peft==0.13.2"

import importlib, peft
# peft was already imported during the environment audit.
# Do not reload it: reloading can leave PEFT/Transformers submodules inconsistent.
print("Using already-loaded peft module:", peft.__file__)
print("peft:", peft.__version__)

# Confirm nothing else moved.
import torch, numpy, diffusers
print("torch:", torch.__version__, "| numpy:", numpy.__version__, "| diffusers:", diffusers.__version__)
assert torch.__version__.startswith("2.1.2"),  "torch drifted - reinstall torch==2.1.2+cu121"
assert numpy.__version__.startswith("1."),     "numpy drifted to 2.x - reinstall numpy<2.0.0"
assert diffusers.__version__ == "0.30.3",      "diffusers drifted - reinstall diffusers==0.30.3"
print("\nenvironment intact")


Using already-loaded peft module: r:\Personal\Projects\DL\NexMarket\backend\.venv\Lib\site-packages\peft\__init__.py
peft: 0.13.2
torch: 2.1.2+cu121 | numpy: 1.26.4 | diffusers: 0.30.3

environment intact


## Cell 3 — Dataset audit  ⚠️ **decision gate**

Before writing a training loop, establish what we would be training *on*.

The thing to look for is **provenance**. If all candidate images were produced by the same SDXL pipeline we are about to fine-tune, then training on them is **self-distillation**: the model is fed its own output, learns nothing it did not already encode, and amplifies its own artifacts. This is the path to model collapse — worth several hours to avoid.

In [16]:
import sqlite3
from pathlib import Path
from collections import Counter

BACKEND = Path("backend")
DB      = BACKEND / "nexmarket.db"

con = sqlite3.connect(str(DB), timeout=15)
con.execute("PRAGMA busy_timeout=15000")

posters = con.execute("SELECT id, provider, image_url, prompt_used FROM posters").fetchall()
by_provider = Counter(p[1] for p in posters)

products = con.execute("""
    SELECT c.product_name, COUNT(*) FROM posters p
    JOIN campaigns c ON c.id = p.campaign_id
    GROUP BY c.product_name ORDER BY 2 DESC
""").fetchall()

n_evals = con.execute("SELECT COUNT(*) FROM poster_evaluations").fetchone()[0]
scores  = [r[0] for r in con.execute("SELECT overall_score FROM poster_evaluations")]
con.close()

print(f"poster rows      : {len(posters)}")
print(f"by provider      : {dict(by_provider)}")
print(f"distinct products: {len(products)}  ->  {products[:3]}")
print(f"evaluations      : {n_evals}  scores={scores}")

# Provenance: anything our own pipeline made is unusable as ground truth.
SELF_GENERATED = {"local-sdxl", "local-sdxl-composed", "openai-dalle3", "mock"}
self_made = sum(v for k, v in by_provider.items() if k in SELF_GENERATED)

print("\n" + "=" * 62)
print(f"self-generated : {self_made}/{len(posters)}")

DATA_IS_USABLE = (
    len(posters) - self_made >= 100 and len(products) >= 10
)

if not DATA_IS_USABLE:
    print("VERDICT: NOT USABLE as SDXL training data (use external dataset).")
else:
    print("VERDICT: usable - proceed.")
print("=" * 62)


poster rows      : 33
by provider      : {'local-sdxl': 19, 'local-sdxl-composed': 14}
distinct products: 2  ->  [('Birla Opus Paints', 17), ('CloudNotes Pro', 5)]
evaluations      : 4  scores=[75.0, 72.0, 65.0, 72.0]

self-generated : 33/33
VERDICT: NOT USABLE as SDXL training data (use external dataset).


## Cell 4 — Load the external dataset

Point at the folder of **external** marketing images (not pipeline output).

```
data/poster_train/
    pexels_0000.jpg
    pexels_0000.txt      <- caption, same stem
    pexels_0001.jpg
    pexels_0001.txt
    ...
```

Captions describe *the visual*, since that is all SDXL produces — subject, environment, lighting, composition, mood. No headlines or CTAs: those are drawn by `PosterComposer`, never by the model.

> `nxmposter, photo of a matte-black water bottle on wet slate, hard rim light from the left, deep shadows, condensation, centred hero product shot, cool teal palette`

**Licensing is on you** — only train on images you have the right to use.

In [17]:
from pathlib import Path

# ── configure ────────────────────────────────────────────────────────────────
DATA_DIR     = Path("data/poster_train")   # external images
OUTPUT_DIR   = Path("backend/ml/lora")     # where the trained LoRA lands
TRIGGER_WORD = "nxmposter"                 # rare token; put it in every caption
RESOLUTION   = 768                         # 1024 is SDXL-native but will OOM on 6GB
# ─────────────────────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXTS = {".png", ".jpg", ".jpeg", ".webp"}

if not DATA_DIR.exists():
    print(f"DATA_DIR does not exist: {DATA_DIR.resolve()}")
    images = []
else:
    images = sorted(p for p in DATA_DIR.iterdir() if p.suffix.lower() in EXTS)

pairs = []
missing_caption = 0
for img in images:
    cap_file = img.with_suffix(".txt")
    if cap_file.exists():
        caption = cap_file.read_text(encoding="utf-8").strip()
    else:
        caption = TRIGGER_WORD
        missing_caption += 1
    if TRIGGER_WORD not in caption:
        caption = f"{TRIGGER_WORD}, {caption}"
    pairs.append((img, caption))

print(f"images found     : {len(pairs)}")
print(f"missing captions : {missing_caption}")
if pairs:
    print("\nsample:")
    for img, cap in pairs[:2]:
        print(f"  {img.name}")
        print(f"    {cap[:100]}...")

if len(pairs) < 20:
    print(f"\n{len(pairs)} images is not enough (need ~20 minimum).")

READY = len(pairs) >= 20
print(f"\nREADY = {READY}")


images found     : 300
missing captions : 0

sample:
  pexels_0000.jpg
    nxmposter, A professional advertisement featuring a clean, minimalist flat-lay composition centered ...
  pexels_0001.jpg
    nxmposter, A professional advertisement featuring a clean, modern flat-lay composition centered on f...

READY = True


## Cell 5 — Pre-compute latents and text embeddings

**This is the step that makes 6 GB work.**

The VAE (~330 MB) and SDXL's *two* text encoders (CLIP ViT-L + OpenCLIP bigG, ~1.4 GB together) are only needed to convert images→latents and captions→embeddings. Doing that once and caching frees them, so only the UNet occupies VRAM during training.

The VAE runs in **fp32**: SDXL's base VAE produces NaNs in fp16.

In [18]:
import gc, torch
from PIL import Image
import torchvision.transforms as T
from diffusers import AutoencoderKL
from transformers import CLIPTextModel, CLIPTextModelWithProjection, AutoTokenizer

assert READY, "Cell 4 reported READY=False - supply images first."

MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"
device   = "cuda"

tf = T.Compose([
    T.Resize(RESOLUTION, interpolation=T.InterpolationMode.BILINEAR),
    T.CenterCrop(RESOLUTION),
    T.ToTensor(),
    T.Normalize([0.5], [0.5]),          # -> [-1, 1]
])

# ── 1. images -> latents (fp32 VAE) ────────────────────────────────────────
# Only the fp16-variant weights are cached locally (no plain
# diffusion_pytorch_model.safetensors was ever downloaded), so variant="fp16"
# is required to find the file at all. torch_dtype=float32 still upcasts the
# loaded weights and runs the forward pass in fp32, which is what actually
# avoids the VAE's NaN overflow - independent of what precision the checkpoint
# was saved in.
vae = AutoencoderKL.from_pretrained(
    MODEL_ID,
    subfolder="vae",
    variant="fp16",
    torch_dtype=torch.float32,
    use_safetensors=True,
    local_files_only=True,
).to(device).eval()

latents = []
with torch.no_grad():
    for i, (img_path, _) in enumerate(pairs):
        if i % 50 == 0: print(f"  encode {i}/{len(pairs)}")
        img = Image.open(img_path).convert("RGB")
        x   = tf(img).unsqueeze(0).to(device, torch.float32)
        lat = vae.encode(x).latent_dist.sample() * vae.config.scaling_factor
        latents.append(lat.squeeze(0).cpu())

print(f"latents: {len(latents)} x {tuple(latents[0].shape)}")

del vae; gc.collect(); torch.cuda.empty_cache()

# ── 2. captions -> embeddings (SDXL needs BOTH encoders) ────────────────────
tok1 = AutoTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer",   local_files_only=True)
tok2 = AutoTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer_2", local_files_only=True)
# text_encoder_2 also only has the fp16-variant file cached; text_encoder
# happens to have both, but pass variant="fp16" on both for consistency.
te1  = CLIPTextModel.from_pretrained(
    MODEL_ID, subfolder="text_encoder",   variant="fp16",
    torch_dtype=torch.float16, local_files_only=True).to(device).eval()
te2  = CLIPTextModelWithProjection.from_pretrained(
    MODEL_ID, subfolder="text_encoder_2", variant="fp16",
    torch_dtype=torch.float16, local_files_only=True).to(device).eval()

prompt_embeds_all, pooled_all = [], []
with torch.no_grad():
    for i, (_, caption) in enumerate(pairs):
        if i % 50 == 0: print(f"  embed {i}/{len(pairs)}")
        i1 = tok1(caption, padding="max_length", max_length=77, truncation=True, return_tensors="pt").input_ids.to(device)
        i2 = tok2(caption, padding="max_length", max_length=77, truncation=True, return_tensors="pt").input_ids.to(device)

        o1 = te1(i1, output_hidden_states=True).hidden_states[-2]
        o2 = te2(i2, output_hidden_states=True)
        pooled = o2.text_embeds
        o2h    = o2.hidden_states[-2]

        prompt_embeds_all.append(torch.cat([o1, o2h], dim=-1).squeeze(0).cpu())
        pooled_all.append(pooled.squeeze(0).cpu())

print(f"prompt_embeds: {tuple(prompt_embeds_all[0].shape)}  pooled: {tuple(pooled_all[0].shape)}")

del te1, te2, tok1, tok2; gc.collect(); torch.cuda.empty_cache()
print(f"\nfree VRAM: {torch.cuda.mem_get_info()[0]/1024**3:.2f} GB")


r:\Personal\Projects\DL\NexMarket\backend\.venv\Lib\site-packages\torch\nn\modules\module.py:1158: UserWarning: expandable_segments not supported on this platform (Triggered internally at ..\c10\cuda\CUDACachingAllocator.cpp:803.)
  return t.to(device, dtype if t.is_floating_point() or t.is_complex() else None, non_blocking)


  encode 0/300
  encode 50/300
  encode 100/300
  encode 150/300
  encode 200/300
  encode 250/300
latents: 300 x (4, 96, 96)
  embed 0/300
  embed 50/300
  embed 100/300
  embed 150/300
  embed 200/300
  embed 250/300
prompt_embeds: (77, 2048)  pooled: (1280,)

free VRAM: 4.90 GB


## Cell 6 — Attach LoRA to the UNet

Only the attention projections (`to_q`, `to_k`, `to_v`, `to_out.0`) get adapters. That is where cross-attention injects the text conditioning, so it is where style adaptation lands — and it keeps the trainable parameter count near ~1 % of the UNet.

Base weights stay frozen in fp16; the LoRA parameters are fp32 for stable gradients.

In [19]:
from diffusers import UNet2DConditionModel, DDPMScheduler
from peft import LoraConfig, get_peft_model

LORA_RANK  = 16     # rank of the low-rank adaptation
LORA_ALPHA = 16     # scaling

# Only the fp16-variant unet weights are cached locally - same reason as the
# VAE fix above.
unet = UNet2DConditionModel.from_pretrained(
    MODEL_ID, subfolder="unet", variant="fp16",
    torch_dtype=torch.float16, local_files_only=True,
).to(device)

unet.requires_grad_(False)              # freeze the base model
unet.enable_gradient_checkpointing()    # trades compute for VRAM

lora_cfg = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    init_lora_weights="gaussian",
    target_modules=["to_q", "to_k", "to_v", "to_out.0"],
)
unet.add_adapter(lora_cfg)

# LoRA params in fp32 - fp16 gradients on rank-16 matrices underflow.
lora_params = [p for p in unet.parameters() if p.requires_grad]
for p in lora_params:
    p.data = p.data.float()

n_train = sum(p.numel() for p in lora_params)
n_total = sum(p.numel() for p in unet.parameters())
print(f"trainable: {n_train:,} / {n_total:,}  ({100*n_train/n_total:.2f}%)")

noise_scheduler = DDPMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler", local_files_only=True)
print(f"free VRAM: {torch.cuda.mem_get_info()[0]/1024**3:.2f} GB")


trainable: 23,224,320 / 2,590,688,004  (0.90%)
free VRAM: 0.00 GB


## Cell 7 — Train

Standard diffusion objective: noise the cached latent to a random timestep, ask the UNet to predict the noise, minimise MSE.

SDXL additionally requires `added_cond_kwargs` with the pooled text embedding plus `time_ids`. Measured on this 3050: ~6 min per 100 steps at 768 px (so 800 steps ≈ 50 min) — VRAM peaks around 5.6/6.0 GB, so close the notebook kernel of anything else GPU-heavy before running this.

In [20]:
import math, random, time
from torch.nn import functional as F

# ── hyperparameters ──────────────────────────────────────────────────────────
MAX_STEPS      = 800     # ~100 imgs: 500-1500 is the useful band
LR             = 1e-4    # LoRA tolerates a far higher LR than a full fine-tune
GRAD_ACCUM     = 4       # effective batch 4 with batch_size 1
SAVE_EVERY     = 200
# ─────────────────────────────────────────────────────────────────────────────

optimizer = torch.optim.AdamW(lora_params, lr=LR, weight_decay=1e-2)
lr_sched  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_STEPS)

# SDXL micro-conditioning: (orig_h, orig_w, crop_top, crop_left, target_h, target_w)
time_ids = torch.tensor(
    [[RESOLUTION, RESOLUTION, 0, 0, RESOLUTION, RESOLUTION]],
    dtype=torch.float16, device=device,
)

unet.train()
N = len(latents)
losses, t0 = [], time.time()

for step in range(1, MAX_STEPS + 1):
    accum_loss = 0.0
    for _ in range(GRAD_ACCUM):
        i = random.randrange(N)

        lat = latents[i].unsqueeze(0).to(device, torch.float16)
        pe  = prompt_embeds_all[i].unsqueeze(0).to(device, torch.float16)
        po  = pooled_all[i].unsqueeze(0).to(device, torch.float16)

        noise     = torch.randn_like(lat)
        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (1,), device=device).long()
        noisy     = noise_scheduler.add_noise(lat, noise, timesteps)

        pred = unet(
            noisy, timesteps, encoder_hidden_states=pe,
            added_cond_kwargs={"text_embeds": po, "time_ids": time_ids},
        ).sample

        loss = F.mse_loss(pred.float(), noise.float()) / GRAD_ACCUM
        loss.backward()
        accum_loss += loss.item()

    torch.nn.utils.clip_grad_norm_(lora_params, 1.0)
    optimizer.step(); lr_sched.step(); optimizer.zero_grad(set_to_none=True)
    losses.append(accum_loss)

    if step % 25 == 0:
        recent = sum(losses[-25:]) / 25
        el     = time.time() - t0
        eta    = el / step * (MAX_STEPS - step)
        print(f"step {step:4d}/{MAX_STEPS}  loss {recent:.4f}  "
              f"lr {lr_sched.get_last_lr()[0]:.2e}  eta {eta/60:.1f}m  "
              f"vram {torch.cuda.max_memory_allocated()/1024**3:.2f}GB")

    if step % SAVE_EVERY == 0 or step == MAX_STEPS:
        from peft.utils import get_peft_model_state_dict
        from safetensors.torch import save_file
        sd  = {k: v.to(torch.float16).cpu() for k, v in get_peft_model_state_dict(unet).items()}
        out = OUTPUT_DIR / "sdxl_poster_lora.safetensors"
        save_file(sd, str(out))
        print(f"  saved {out}  ({sum(v.numel() for v in sd.values()):,} params)")

print(f"\ndone in {(time.time()-t0)/60:.1f} min")


step   25/800  loss 0.1103  lr 9.98e-05  eta 104.9m  vram 5.67GB
step   50/800  loss 0.1177  lr 9.90e-05  eta 102.0m  vram 5.67GB
step   75/800  loss 0.1268  lr 9.78e-05  eta 98.8m  vram 5.67GB
step  100/800  loss 0.1132  lr 9.62e-05  eta 95.7m  vram 5.67GB
step  125/800  loss 0.1187  lr 9.41e-05  eta 92.1m  vram 5.67GB
step  150/800  loss 0.1097  lr 9.16e-05  eta 88.6m  vram 5.67GB
step  175/800  loss 0.1319  lr 8.87e-05  eta 85.1m  vram 5.67GB
step  200/800  loss 0.1017  lr 8.54e-05  eta 81.6m  vram 5.67GB
  saved backend\ml\lora\sdxl_poster_lora.safetensors  (23,224,320 params)
step  225/800  loss 0.1088  lr 8.17e-05  eta 78.2m  vram 5.67GB
step  250/800  loss 0.1012  lr 7.78e-05  eta 74.7m  vram 5.67GB
step  275/800  loss 0.1168  lr 7.36e-05  eta 71.3m  vram 5.67GB
step  300/800  loss 0.1253  lr 6.91e-05  eta 67.9m  vram 5.67GB
step  325/800  loss 0.1011  lr 6.45e-05  eta 64.5m  vram 5.67GB
step  350/800  loss 0.1095  lr 5.98e-05  eta 61.0m  vram 5.67GB
step  375/800  loss 0.1123  

## Cell 8 — Compare before / after

The only honest test is the same prompt and the same **seed**, with and without the adapter. Anything else confounds the LoRA's effect with sampling noise.

In [21]:
import gc, torch
from diffusers import StableDiffusionXLPipeline

del unet; gc.collect(); torch.cuda.empty_cache()

TEST_PROMPT = f"{TRIGGER_WORD}, marketing visual for a premium water bottle, studio lighting, centred hero shot"
TEST_SEED   = 12345

pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, use_safetensors=True,
    variant="fp16", local_files_only=True,
)
pipe.enable_model_cpu_offload()
pipe.enable_vae_slicing()

def render(tag):
    g = torch.Generator(device="cpu").manual_seed(TEST_SEED)
    img = pipe(TEST_PROMPT, num_inference_steps=28, guidance_scale=7.5,
               width=RESOLUTION, height=RESOLUTION, generator=g).images[0]
    img.save(f"compare_{tag}.png")
    return img

before = render("before")

lora_file = OUTPUT_DIR / "sdxl_poster_lora.safetensors"
if lora_file.exists():
    pipe.load_lora_weights(str(lora_file))
    after = render("after")
    print("Comparison saved: compare_before.png vs compare_after.png")
else:
    print(f"No LoRA found at {lora_file}")


100%|██████████| 28/28 [02:53<00:00,  6.21s/it]


Comparison saved: compare_before.png vs compare_after.png


## Cell 9 — Integrate into the backend

Once the comparison above is genuinely better, wire it in. The change is small and additive: load the adapter after the pipeline is built, only if the file exists.

**In `backend/ai/vision/sdxl_service.py`** at the end of `_do_load()` — after the memory optimisations, before `self._pipeline = pipe`:

```python
# Optional fine-tuned poster LoRA. Absent by default; the base model is used
# unchanged when the file is not present.
lora_path = Path(settings.SDXL_LORA_PATH)
if lora_path.exists():
    pipe.load_lora_weights(str(lora_path))
    logger.info("sdxl lora loaded path=%s", lora_path)
```

**In `backend/app/core/config.py`**:

```python
SDXL_LORA_PATH: str = "backend/ml/lora/sdxl_poster_lora.safetensors"
```

**In `requirements.txt`** add:

```
peft==0.13.2
```

**In `backend/app/services/ai/poster_generator.py`** — the prompt must include the trigger word:

In `_build_spec_prompt`, prepend `TRIGGER_WORD`:

```python
spec = f"{TRIGGER_WORD}, {spec}"
```

Then verify with the existing test suite:

```bash
pytest tests/test_main.py::test_poster_generation -v
```
